# Greengard 2023 Table 3 & Table 4 (Aligned Data Check)

This notebook reuses the data configuration from `gpu/sanity_check/v1_mat32_eps_topq_sanity.ipynb`
so you can compare CPU EigenPro behavior on the same dataset.
We keep the table-style output but use the Mat32 + random dataset settings from the V1/V2 sanity check.

In [8]:
import sys
import time
import math
import csv
import importlib
from pathlib import Path

import numpy as np

_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for base in _candidates:
    if (base / "efgp_eigenpro_py").exists():
        if str(base) not in sys.path:
            sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

import efgp_eigenpro_py.efgp_solver as efgp_solver
import efgp_eigenpro_py.toeplitz as toeplitz
import efgp_eigenpro_py.eigenspace as eigenspace
importlib.reload(toeplitz)
importlib.reload(efgp_solver)
importlib.reload(eigenspace)

from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py import benchmark as bm
from efgp_eigenpro_py.benchmark import make_dataset, make_test_set, true_func_2d
importlib.reload(bm)

np.set_printoptions(precision=4, suppress=True)

In [9]:
# Align with v1_mat32_eps_topq_sanity.ipynb
REG_LAMBDA = 0.1
N_TRAIN = 15_000_00
DIM = 2
VARIANCE = 1.0
LENGTHSCALE = 0.1
NU_MATERN = 1.5
KERNEL = make_matern(lengthscale=LENGTHSCALE, dim=DIM, nu=NU_MATERN, variance=VARIANCE)
KERNEL_LABEL = "Mat32"

N_TEST = 20_000
NOISE = 0.02
SEED_TRAIN = 0
SEED_TEST = 1

N_LIST = [N_TRAIN]
EPS_LIST = [1e-5, 1e-6]
EPS_REF = 1e-8
COMPUTE_EEPM = False
EEPM_TRAIN_MAX_N = None
EEPM_TRAIN_TOPQ = None

TOP_Q_SWEEP = [0, 4, 8, 16]
CG_MAXITER_BASELINE = 3000
CG_MAXITER_PRECOND = 3000
CG_MAXITER_REF = 3000

EIG_METHOD = "subspace_iter"
EIG_TOL = 1e-2
EIG_MAXITER = 20
EIG_BLOCK_SIZE = None
EIG_OVERSAMPLE = 2

L2_SCALED = True
MEAN_INCLUDE_TRAIN = False
SIGMA_MODE = "v1"

ENABLE_SOLVE_BREAKDOWN = True

CSV_OUTPUT_DIR = Path("results")
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH_TABLE3 = CSV_OUTPUT_DIR / "greengard_table3_breakdown.csv"
CSV_PATH_TABLE4 = CSV_OUTPUT_DIR / "greengard_table4_breakdown.csv"

print("N_TRAIN=", N_TRAIN, "N_TEST=", N_TEST, "TOP_Q_SWEEP=", TOP_Q_SWEEP)

N_TRAIN= 1500000 N_TEST= 20000 TOP_Q_SWEEP= [0, 4, 8, 16]


In [10]:
def rms(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a - b) ** 2)))


def sigma_for_N(_n_value: int, _mode: str) -> float:
    return float(NOISE)


def seed_for_case(_n_value: int, _mode: str) -> int:
    return int(SEED_TRAIN)


def make_targets(_seed: int, _sigma: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    return x_test, y_test, y_test


print("Building dataset ...")
x_train, y_train = make_dataset(DIM, N_TRAIN, true_func_2d, noise=NOISE, seed=SEED_TRAIN)
x_test, y_test = make_test_set(DIM, N_TEST, true_func_2d, seed=SEED_TEST)

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)
print("kernel:", KERNEL)
print("targets shape=", x_test.shape)

Building dataset ...
x_train: (1500000, 2) float64
y_train: (1500000,) float64
x_test : (20000, 2) float64
kernel: KernelSpec(fam='matern', dim=2, lengthscale=0.1, variance=1.0, nu=1.5)
targets shape= (20000, 2)


In [11]:
def precompute_streaming_case(_n_value: int, _sigma: float, eps: float, _seed: int):
    reg_lambda = float(REG_LAMBDA)
    nufft_eps = eps / 10.0
    solver = EFGPSolver(
        KERNEL,
        reg_lambda=reg_lambda,
        eps=eps,
        nufft_tol=nufft_eps,
        l2scaled=L2_SCALED,
    )
    t0 = time.perf_counter()
    state = solver.precompute(x_train, y_train)
    t1 = time.perf_counter()
    pre_times = {"time_precompute_total": t1 - t0}
    return solver, state, pre_times


def compute_train_eepm(
    solver: EFGPSolver,
    state,
    beta: np.ndarray,
    n_value: int,
    _sigma: float,
    _seed: int,
    *,
    max_n: int | None = None,
) -> float:
    n_eval = int(n_value) if max_n is None else int(min(n_value, max_n))
    if n_eval <= 0:
        return float("nan")
    n_eval = min(n_eval, x_train.shape[0])
    x_eval = x_train[:n_eval]
    f_eval = true_func_2d(x_eval)
    yhat = solver.predict(x_eval, beta, state)
    return float(np.sqrt(np.mean((yhat - f_eval) ** 2)))


def _solve_breakdown_header() -> str:
    return (
        "solve_s_total n_iter n_matvec t_matvec_total t_matvec_avg "
        "n_precond t_precond_total t_precond_avg t_cg_overhead"
    )


def print_table_header():
    header = (
        "sigma N d kernel eps top_q m pre_s eig_s precond_s solve_s mean_s total_s "
        "iters relres converged hit_max eepm_train eepm_new rms rmse_ex"
    )
    if ENABLE_SOLVE_BREAKDOWN:
        header = header + " " + _solve_breakdown_header()
    print(header)


def format_table_row(row: dict) -> str:
    converged = "Y" if row["converged"] else "N"
    hit_maxiter = "Y" if row["hit_maxiter"] else "N"
    base = (
        f"{row['sigma']:>8.2e} {row['N']:>10d} {row['dim']:>1d} {row['kernel']:>7s} {row['eps']:>8.1e} "
        f"{row['top_q']:>5d} {row['m']:>4d} {row['pre']:>8.3f} {row['eig']:>8.3f} "
        f"{row['precond']:>8.3f} {row['solve']:>8.3f} {row['mean']:>8.3f} {row['total']:>8.3f} "
        f"{row['iters']:>6d} {row['relres']:>8.2e} {converged:>9s} {hit_maxiter:>7s} "
        f"{row['eepm_train']:>9.2e} {row['eepm_new']:>9.2e} {row['rms']:>9.2e} {row['rmse_ex']:>9.2e}"
    )
    if not ENABLE_SOLVE_BREAKDOWN:
        return base
    return base + (
        f" {row['solve_s_total']:>9.3f} {row['n_iter']:>6d} {row['n_matvec']:>8d} "
        f"{row['t_matvec_total']:>11.3f} {row['t_matvec_avg']:>11.3e} {row['n_precond']:>8d} "
        f"{row['t_precond_total']:>13.3f} {row['t_precond_avg']:>12.3e} {row['t_cg_overhead']:>12.3f}"
    )

In [12]:
cache = {}
ref_cache = {}


def get_case_entry(n_value: int, sigma: float, eps: float, seed: int) -> dict:
    key = (int(n_value), float(sigma), float(eps))
    if key in cache:
        return cache[key]
    solver, state, pre_times = precompute_streaming_case(n_value, sigma, eps, seed)
    entry = {
        "N": int(n_value),
        "sigma": float(sigma),
        "eps": float(eps),
        "solver": solver,
        "state": state,
        "pre": float(pre_times["time_precompute_total"]),
    }
    cache[key] = entry
    return entry


def compute_reference_pred(n_value: int, sigma: float, seed: int, x_trg: np.ndarray) -> dict:
    key = (int(n_value), float(sigma), float(EPS_REF))
    if key in ref_cache:
        return ref_cache[key]
    solver, state, _pre_times = precompute_streaming_case(n_value, sigma, EPS_REF, seed)
    matvec = lambda v: solver._apply_A(state, v)
    beta, it, relres, _hist = bm.pcg_solve(
        matvec,
        state.rhs,
        tol=EPS_REF,
        maxiter=CG_MAXITER_REF,
        precond=None,
    )
    yhat_ref = solver.predict(x_trg, beta, state)
    entry = {
        "yhat_ref": yhat_ref,
        "iters": int(it),
        "relres": float(relres),
        "converged": bool(relres <= EPS_REF),
        "hit_maxiter": bool(it >= CG_MAXITER_REF),
    }
    ref_cache[key] = entry
    return entry


def solve_with_topq(entry: dict, top_q: int, x_trg: np.ndarray, y_trg_test: np.ndarray, ref=None) -> dict:
    solver = entry["solver"]
    state = entry["state"]
    eps = entry["eps"]
    matvec = lambda v: solver._apply_A(state, v)

    precond, _eigpairs, _mu, t_eig, t_precond, top_q_used = bm.build_precond_from_state(
        solver,
        state,
        top_q,
        eig_method=EIG_METHOD,
        eig_tol=EIG_TOL,
        eig_maxiter=EIG_MAXITER,
        eig_block_size=EIG_BLOCK_SIZE,
        eig_oversample=EIG_OVERSAMPLE,
    )
    precond_apply = precond.apply if precond is not None else None

    maxiter = CG_MAXITER_BASELINE if top_q_used == 0 else CG_MAXITER_PRECOND

    t_s0 = time.perf_counter()
    if ENABLE_SOLVE_BREAKDOWN:
        beta, it, relres, _hist, stats = bm.pcg_solve(
            matvec,
            state.rhs,
            tol=eps,
            maxiter=maxiter,
            precond=precond_apply,
            store_history=False,
            return_stats=True,
        )
    else:
        beta, it, relres, _hist = bm.pcg_solve(
            matvec,
            state.rhs,
            tol=eps,
            maxiter=maxiter,
            precond=precond_apply,
            store_history=False,
        )
        stats = None
    t_s1 = time.perf_counter()

    t_m0 = time.perf_counter()
    yhat = solver.predict(x_trg, beta, state)
    t_m1 = time.perf_counter()

    err_rms = rms(y_trg_test, yhat)
    if ref is None:
        err_eepm_new = float("nan")
        err_rmse_ex = float("nan")
        err_eepm_train = float("nan")
    else:
        yhat_ref = ref["yhat_ref"]
        err_eepm_new = rms(yhat, yhat_ref)
        err_rmse_ex = rms(y_trg_test, yhat_ref)
        compute_train = True
        if EEPM_TRAIN_TOPQ is not None and top_q_used not in EEPM_TRAIN_TOPQ:
            compute_train = False
        if EEPM_TRAIN_MAX_N is not None and entry["N"] > EEPM_TRAIN_MAX_N:
            compute_train = False
        if compute_train:
            err_eepm_train = compute_train_eepm(
                solver,
                state,
                beta,
                entry["N"],
                entry["sigma"],
                seed_for_case(entry["N"], SIGMA_MODE),
                max_n=EEPM_TRAIN_MAX_N,
            )
        else:
            err_eepm_train = float("nan")

    solve_time = t_s1 - t_s0
    mean_time = t_m1 - t_m0
    pre = entry["pre"]
    total = pre + t_eig + t_precond + solve_time + mean_time
    m = (state.grid.mtot - 1) // 2

    row = {
        "sigma": entry["sigma"],
        "N": entry["N"],
        "dim": int(KERNEL.dim),
        "kernel": KERNEL_LABEL,
        "eps": float(eps),
        "top_q": int(top_q_used),
        "m": int(m),
        "pre": float(pre),
        "eig": float(t_eig),
        "precond": float(t_precond),
        "solve": float(solve_time),
        "mean": float(mean_time),
        "total": float(total),
        "iters": int(it),
        "relres": float(relres),
        "converged": bool(relres <= eps),
        "hit_maxiter": bool(it >= maxiter),
        "eepm_train": float(err_eepm_train),
        "eepm_new": float(err_eepm_new),
        "rms": float(err_rms),
        "rmse_ex": float(err_rmse_ex),
    }

    if ENABLE_SOLVE_BREAKDOWN:
        n_matvec = stats["n_matvec"]
        t_matvec_total = stats["t_matvec_total"]
        n_precond = stats["n_precond"]
        t_precond_total = stats["t_precond_total"]
        t_matvec_avg = t_matvec_total / max(n_matvec, 1)
        t_precond_avg = t_precond_total / max(n_precond, 1)
        t_cg_overhead = solve_time - t_matvec_total - t_precond_total
        row.update(
            {
                "solve_s_total": float(solve_time),
                "n_iter": int(it),
                "n_matvec": int(n_matvec),
                "t_matvec_total": float(t_matvec_total),
                "t_matvec_avg": float(t_matvec_avg),
                "n_precond": int(n_precond),
                "t_precond_total": float(t_precond_total),
                "t_precond_avg": float(t_precond_avg),
                "t_cg_overhead": float(t_cg_overhead),
            }
        )

    return row


def _write_csv(rows: list[dict], path: Path):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def run_table(
    sigma_mode: str,
    n_list: list[int] | None = None,
    eps_list: list[float] | None = None,
    top_q_list: list[int] | None = None,
    print_rows: bool = True,
    max_rows: int | None = None,
    return_on_interrupt: bool = True,
    csv_path: Path | None = None,
):
    if n_list is None:
        n_list = list(N_LIST)
    if eps_list is None:
        eps_list = list(EPS_LIST)
    if top_q_list is None:
        top_q_list = list(TOP_Q_SWEEP)

    rows = []
    if print_rows:
        print_table_header()

    last_case = None
    try:
        for n_value in n_list:
            sigma = sigma_for_N(n_value, sigma_mode)
            seed = seed_for_case(n_value, sigma_mode)
            x_trg, y_trg_test, _y_true = make_targets(seed + 999, sigma)
            ref = compute_reference_pred(n_value, sigma, seed, x_trg) if COMPUTE_EEPM else None

            for eps in eps_list:
                entry = get_case_entry(n_value, sigma, eps, seed)
                for top_q in top_q_list:
                    last_case = (n_value, sigma, eps, top_q)
                    row = solve_with_topq(entry, top_q, x_trg, y_trg_test, ref=ref)
                    rows.append(row)
                    if print_rows:
                        print(format_table_row(row))
                    if max_rows is not None and len(rows) >= max_rows:
                        if csv_path is not None:
                            _write_csv(rows, csv_path)
                        return rows
    except KeyboardInterrupt:
        if return_on_interrupt:
            if last_case is not None:
                n_value, sigma, eps, top_q = last_case
                print(
                    "Interrupted at N={}, sigma={:.2e}, eps={:.1e}, top_q={}. Returning partial rows.".format(
                        n_value, sigma, eps, top_q
                    )
                )
            if csv_path is not None:
                _write_csv(rows, csv_path)
            return rows
        raise

    if csv_path is not None:
        _write_csv(rows, csv_path)
    return rows

In [13]:
# Table 3: fixed sigma
SIGMA_MODE = "table3"
rows_table3 = run_table(SIGMA_MODE, csv_path=CSV_PATH_TABLE3)

# Optional: make a DataFrame
# import pandas as pd
# df_table3 = pd.DataFrame(rows_table3)
# df_table3

sigma N d kernel eps top_q m pre_s eig_s precond_s solve_s mean_s total_s iters relres converged hit_max eepm_train eepm_new rms rmse_ex solve_s_total n_iter n_matvec t_matvec_total t_matvec_avg n_precond t_precond_total t_precond_avg t_cg_overhead
2.00e-02    1500000 2   Mat32  1.0e-05     0   94    0.344    0.000    0.000    2.603    0.027    2.974    372 9.13e-06         Y       N       nan       nan  1.11e-03       nan     2.603    372      373       2.383   6.389e-03        0         0.000    0.000e+00        0.220
2.00e-02    1500000 2   Mat32  1.0e-05     4   94    0.344    1.122    0.001    2.821    0.009    4.297    336 8.75e-06         Y       N       nan       nan  1.11e-03       nan     2.821    336      337       2.227   6.607e-03      336         0.383    1.139e-03        0.212
2.00e-02    1500000 2   Mat32  1.0e-05     8   94    0.344    1.784    0.001    2.815    0.011    4.955    298 8.43e-06         Y       N       nan       nan  1.11e-03       nan     2.815    298   